# 数据结构与科学计算 · 第 04 讲
## 链表、动态数组、引用与环形缓冲区

> **课程定位**：面向已经学习 Python 基础、正在准备自行编写二维 CFD 求解器的学习者。本讲从数据结构原理出发，以实际数值计算问题作为应用练习。

**完成本讲后，你应该能够：**

1. 理解单向链表的节点、指针引用和遍历；
2. 写出链表头部插入、删除、反转的成本；
3. 理解动态数组容量增长为何使尾部插入成为**摊销 $O(1)$**；
4. 区分 Python `list` 的引用数组与 NumPy 的同类型数值缓冲区；
5. 实现 CFD / 强化学习日志常用的固定容量环形缓冲区。

**预计学习时间**：80–120 分钟。所有核心示例可在 VS Code 的 Jupyter Notebook 中直接运行。

## 0. 使用方法与运行环境

1. 在 VS Code 打开此 `.ipynb`，右上角选择当前项目的 Python Kernel；
2. 使用 `Shift + Enter` 依次运行。修改代码后，建议 **Restart Kernel & Run All** 以消除旧变量干扰；
3. **练习单元以 `TODO` 留空**，后面会有逐项自动检查；未完成练习时显示“尚未通过”是预期行为；
4. 最后附有独立命名的 `_solution` 参考实现，不会覆盖你的练习函数；
5. 学习重点是“选择什么结构、为什么、代价是什么”，不只是背诵 API。

前两讲已安装 NumPy。本系列推荐运行 `uv pip install numpy scipy matplotlib ipykernel jupyter`。第 08 讲还需要 JAX：`uv pip install -U jax`（此命令默认安装适用于当前平台的发行版，GPU 支持视平台而定）。

In [1]:
import sys
print('Python:', sys.version.split()[0])
print('Notebook 已连接到 Python Kernel')

Python: 3.13.5
Notebook 已连接到 Python Kernel


## 1. 把两种顺序容器放在一起看

**数组式存储**：元素（或元素引用）占据连续编号的位置，按下标访问快。**链表**：每个节点除了存储值，还保存下一节点的引用。

```text
数组式:  [A][B][C][D]          随机索引方便
链表式:  A -> B -> C -> D -> None  连接关系方便
```

| 操作 | Python `list`（动态数组） | 单链表（已知头结点） |
|:--|:--|:--|
| 按索引访问 | $O(1)$ | $O(n)$ |
| 头部插入 | $O(n)$ | $O(1)$ |
| 尾部插入 | 摊销 $O(1)$ | 若无尾指针则 $O(n)$ |
| 删除已知头节点 | $O(n)$ | $O(1)$ |
| 搜索指定值 | $O(n)$ | $O(n)$ |

注：上表是抽象操作复杂度；Python 内置 `list` 没有“节点指针”API。链表额外引用意味着缓存局部性通常更差，数值向量化也不方便。

In [2]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Node:
    value: object
    next: Optional['Node'] = None

head = Node('A', Node('B', Node('C')))
current = head
while current is not None:
    print(current.value, end=' -> ')
    current = current.next
print('None')

A -> B -> C -> None


## 2. “引用”与“值”是两个不同层次

下面修改链表链接，**不会**复制整个链表。在数值程序里，类似的引用共享也会使意外修改成为难找的错误，因此需要理解对象别名。

In [3]:
a = Node(1)
b = Node(2)
a.next = b
alias = a
alias.value = 99
print('same object?', a is alias, 'value:',a.value)

# 比较数组共享：NumPy 切片常常是 view
import numpy as np
x = np.array([1, 2, 3, 4])
y = x[1:3]
y[0] = 77
print('x after view modification:', x)

same object? True value: 99
x after view modification: [ 1 77  3  4]


### 2.1 封装一个最小链表

维护 `head`；`push_front` 更新的只是头指针，`to_list` 需要顺着 `next` 遍历所有节点。

In [4]:
class SinglyLinkedList:
    def __init__(self):
        self.head = None

    def push_front(self, value):
        self.head = Node(value, self.head)

    def pop_front(self):
        if self.head is None:
            raise IndexError('pop from empty linked list')
        value = self.head.value
        self.head = self.head.next
        return value

    def to_list(self):
        result, current = [], self.head
        while current is not None:
            result.append(current.value)
            current = current.next
        return result

links = SinglyLinkedList()
for v in [1, 2, 3]:
    links.push_front(v)
print(links.to_list(), links.pop_front(), links.to_list())

[3, 2, 1] 3 [2, 1]


## 3. 动态数组为何尾部追加很快？

如果数组每次追加都重新分配一块恰好增大一个槽位的内存，会非常昂贵。现代动态数组通常**预留额外容量**；只有容量不足时才扩容并搬迁已有元素。

假设教学模型每次容量翻倍，追加 $n$ 个元素的复制总量至多是

$$1+2+4+\cdots+2^k < 2n,$$

于是 $n$ 次追加的均摊总操作量为 $O(n)$，每次为 **摊销 $O(1)$**。这并不表示每一次 `append` 都耗时相同。

下列是**教学用容量翻倍模拟器**，不是 CPython 的真实增长策略。

In [5]:
class ToyDynamicArray:
    def __init__(self):
        self.capacity = 1
        self.values = [None] * self.capacity
        self.length = 0
        self.copies = 0

    def append(self, value):
        if self.length == self.capacity:
            new = [None] * (self.capacity * 2)
            for i in range(self.length):
                new[i] = self.values[i]
                self.copies += 1
            self.values = new
            self.capacity *= 2
        self.values[self.length] = value
        self.length += 1

arr = ToyDynamicArray()
for i in range(1_000):
    arr.append(i)
print('length:',arr.length,'capacity:',arr.capacity,'total copies:',arr.copies)
assert arr.copies < 2 * arr.length

length: 1000 capacity: 1024 total copies: 1023


### 3.1 看 Python `list` 的内存跳变

`sys.getsizeof` 观察的是 **list 容器本体**（包括引用槽位等），不包括列表中所有对象的递归内存占用。不要把结果当成 NumPy `ndarray.nbytes`。

In [6]:
import sys
values = []
records = []
old_size = None
for i in range(100):
    values.append(i)
    current_size = sys.getsizeof(values)
    if current_size != old_size:
        records.append((len(values), current_size))
        old_size = current_size
print('几个长度变化点 (len, list 容器字节数):', records[:12])
print('NumPy 100个float64数值字节数:',np.zeros(100, dtype=np.float64).nbytes)

几个长度变化点 (len, list 容器字节数): [(1, 88), (5, 120), (9, 184), (17, 248), (25, 312), (33, 376), (41, 472), (53, 568), (65, 664), (77, 792), (93, 920)]
NumPy 100个float64数值字节数: 800


## 4. 环形缓冲区：固定空间复用

假设每一步 CFD 迭代都产生残差 $r_t$。只需要最近 $K$ 步日志，不需要保存全部历史。环形缓冲区用写指针循环覆盖旧槽位：

$$\text{write\_index} = (\text{write\_index}+1) \bmod K.$$

Python `collections.deque(maxlen=K)` 也提供自动保留最近 K 项的功能；自行编写环形缓冲区能帮助你理解 RL replay buffer 的存储逻辑。

In [7]:
from collections import deque
recent = deque(maxlen=4)
for r in [1.0, 0.5, 0.25, 0.125, 0.0625]:
    recent.append(r)
print('最近 4 次残差:',list(recent))

class RingBuffer:
    def __init__(self, capacity):
        if capacity <= 0:
            raise ValueError('capacity must be > 0')
        self.data = [None] * capacity
        self.capacity = capacity
        self.count = 0
        self.next_write = 0

    def append(self, value):
        self.data[self.next_write] = value
        self.next_write = (self.next_write + 1) % self.capacity
        self.count = min(self.count + 1, self.capacity)

    def to_list(self):
        start = (self.next_write - self.count) % self.capacity
        return [self.data[(start + k) % self.capacity] for k in range(self.count)]

ring = RingBuffer(3)
for v in ['a','b','c','d']:
    ring.append(v)
print('ring chronological:', ring.to_list())

最近 4 次残差: [0.5, 0.25, 0.125, 0.0625]
ring chronological: ['b', 'c', 'd']


## 5. 在 CFD / RL 中的选择

- **二维速度场**：NumPy/JAX 同类型数组，避免每格一个 Python 节点；
- **频繁对头尾操作的任务流**：`deque`；
- **邻居链式结构**：个别图算法会用链接思想，但 Python 实战中列表/字典通常更方便；
- **最近 K 步残差**：`deque(maxlen=K)`；
- **随机采样的强化学习经验**：预分配 NumPy 数组并通过环形下标覆写，可高效地批量取样。

`list` 里存储的是对象引用，并不是把任意 Python 对象都像 `float64` 数值一样连续存放。

---
## 6. 编程练习

**练习 1：** 写 `linked_to_list(head)`，遍历给定 `Node` 链表的所有元素；空链表返回空列表。

In [8]:
def linked_to_list(head):
    # TODO: 不要修改链表本身
    return None

**练习 2：** 实现 `reverse_linked(head)`，**原地反转**单链表的链接方向，返回新头节点；不要创建新的 Node 节点。提示：保存 `next_node`，否则会丢失后续链。

In [9]:
def reverse_linked(head):
    # TODO: prev, current, next_node
    return None

**练习 3：** 写 `recent_k(stream, k)`，按原顺序返回流中最近 `k` 项，使用固定大小队列或环形缓冲区，`k` 要求大于零，不能修改输入。

In [10]:
def recent_k(stream, k):
    # TODO: 尝试 deque(maxlen=k)
    return None

## 练习自测

运行下方测试。`TODO` 未完成时出现“尚未通过”属于正常情况；逐项填入你的实现并再次运行。本讲的测试检查正确性与边界条件，性能优化可在通过后继续尝试。

In [11]:
def check(name, test_fn):
    try:
        test_fn()
        print(f"✅ {name}：通过")
    except Exception as exc:
        print(f"⬜ {name}：尚未通过 — {type(exc).__name__}: {exc}")

In [12]:
def make_chain(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def test_ex1():
    assert linked_to_list(None) == []
    assert linked_to_list(make_chain([1,2,3])) == [1,2,3]

def test_ex2():
    head = make_chain([1,2,3])
    old_nodes = [head, head.next, head.next.next]
    reversed_head = reverse_linked(head)
    assert reversed_head is old_nodes[-1], '必须复用旧节点'
    assert linked_to_list(reversed_head) == [3,2,1]
    assert old_nodes[0].next is None
    assert reverse_linked(None) is None

def test_ex3():
    seq = [1,2,3,4,5]
    assert recent_k(seq,3) == [3,4,5]
    assert recent_k(seq,10) == seq
    assert recent_k([],4) == []
    assert seq == [1,2,3,4,5]
    try:
        recent_k(seq,0)
    except ValueError:
        pass
    else:
        assert False,'k=0 应抛出 ValueError'

check('练习 1 / 链表遍历',test_ex1)
check('练习 2 / 原地反转',test_ex2)
check('练习 3 / 环形历史',test_ex3)

⬜ 练习 1 / 链表遍历：尚未通过 — AssertionError: 
⬜ 练习 2 / 原地反转：尚未通过 — AssertionError: 必须复用旧节点
⬜ 练习 3 / 环形历史：尚未通过 — AssertionError: 


## 7. 参考答案与验证

In [13]:
def linked_to_list_solution(head):
    result = []
    while head is not None:
        result.append(head.value)
        head = head.next
    return result

def reverse_linked_solution(head):
    prev, current = None, head
    while current is not None:
        next_node = current.next
        current.next = prev
        prev, current = current, next_node
    return prev

def recent_k_solution(stream, k):
    if k <= 0:
        raise ValueError('k must be positive')
    d = deque(maxlen=k)
    for item in stream:
        d.append(item)
    return list(d)

assert linked_to_list_solution(reverse_linked_solution(make_chain([1,2,3]))) == [3,2,1]
assert recent_k_solution(range(10),3) == [7,8,9]
print('✅ 参考答案自测通过')

✅ 参考答案自测通过


## 8. 综合实验：预分配数值型 replay buffer

这一段是简化演示，仅存储状态、动作、奖励，不涉及 `done` 标志、完整训练与持久化。体验为什么预分配二维数组比每步构造复杂 Python 对象更适合批量机器学习。

In [14]:
class NumericReplay:
    def __init__(self, capacity, state_dim):
        self.states = np.empty((capacity,state_dim), dtype=np.float32)
        self.actions = np.empty(capacity, dtype=np.int32)
        self.rewards = np.empty(capacity, dtype=np.float32)
        self.capacity = capacity
        self.size = 0
        self.ptr = 0

    def add(self, state, action, reward):
        self.states[self.ptr] = state
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size+1, self.capacity)

    def sample(self, n, rng):
        if n > self.size:
            raise ValueError('not enough data')
        indices = rng.choice(self.size, size=n, replace=False)
        return self.states[indices], self.actions[indices], self.rewards[indices]

buffer = NumericReplay(5,2)
for t in range(8):
    buffer.add([t,t*t], t%3, float(-t))
states, actions, rewards = buffer.sample(3, np.random.default_rng(7))
print('sampled state shape:', states.shape, 'sample rewards:', rewards)

sampled state shape: (3, 2) sample rewards: [-7. -3. -4.]


## 9. 本讲自测

1. 为什么链表头部插入可以 $O(1)$，但按索引查找却要 $O(n)$？
2. 为什么动态数组追加是**摊销** $O(1)$，而非每次严格 $O(1)$？
3. `sys.getsizeof(list)` 为什么不代表列表递归占用的全部内存？
4. 设计一个容量为 K 的 ring buffer，需要哪些状态变量？
5. CFD 场变量为什么不适合使用单链表作为主要数值存储？

**下一讲**：树、堆与优先队列，为理解自适应网格与任务优先级做准备。